In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import neurokit2 as nk

# 입력/출력 경로
ROOT_DIR = Path("./raw_data")                 # 상위 폴더 (하위에 p01, p02, …)
OUT_DIR  = Path("./dataset/dl_dataset")       # 저장 폴더
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 샘플링/윈도
FS = 128
WIN_SEC  = 5.0
OVERLAP  = 0.5
HOP_SEC  = WIN_SEC * (1.0 - OVERLAP)
WIN = int(round(WIN_SEC * FS))   # 640
HOP = int(round(HOP_SEC * FS))   # 320


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import neurokit2 as nk

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 200)

def read_two_header_as_standard_df(path, sep=None):
    """Shimmer 2줄 헤더(csv/tsv) → 표준컬럼(timestamp, ppg, gsr)로 변환."""
    NAME_TS  = "Shimmer_9F46_TimestampSync_Unix_CAL"
    NAME_PPG = "Shimmer_9F46_PPG_A13_CAL"
    NAME_EDA = "Shimmer_9F46_GSR_Skin_Conductance_CAL"
    UNIT_TS, UNIT_PPG, UNIT_EDA = "ms", "mV", "uS"

    if sep is None:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            first = f.readline()
        sep = "\t" if "\t" in first else ","

    df0 = pd.read_csv(path, sep=sep, header=[0, 1], engine="python")
    df0.columns = pd.MultiIndex.from_tuples([(str(a).strip(), str(b).strip()) for a, b in df0.columns])
    df = pd.DataFrame({
        "timestamp": pd.to_numeric(df0[(NAME_TS,  UNIT_TS)],  errors="coerce"),
        "ppg":       pd.to_numeric(df0[(NAME_PPG, UNIT_PPG)], errors="coerce"),
        "gsr":       pd.to_numeric(df0[(NAME_EDA, UNIT_EDA)], errors="coerce"),
    })
    return df.dropna(subset=["timestamp", "ppg", "gsr"]).reset_index(drop=True)

def _parse_filename_meta(path_str: str):
    """
    파일명 규약 예: p01_34_0_arousal_0_21_0.csv
    parts: 0=p01, 1=trial, 2=file_state, 3='arousal', 4=pre, 5=content_id, 6=post
    """
    name = Path(path_str).stem
    parts = name.split("_")
    if len(parts) < 7:
        return {}
    return {
        "subject":      parts[0],
        "trial":        int(parts[1]),
        "file_state":   int(parts[2]),
        "pre_arousal":  int(parts[4]),
        "content_id":   int(parts[5]),
        "post_arousal": int(parts[6]),
    }

def trim_windows_by_state(windows_idx, file_state: int, target_n: int = 25):
    """
    file_state:
      0=그대로,
      1=앞 keep,
      2=뒤 keep,
      3=중앙 keep (개수=target_n),
      4=그대로 (0처럼),
      5=앞에서 target_n만큼 자르되, 부족하면 있는 만큼만 반환
    """
    n = len(windows_idx)
    if n == 0 or file_state in [0, 4]:
        return windows_idx
    keep = min(target_n, n)
    if file_state == 1:
        return windows_idx[:keep]
    elif file_state == 2:
        return windows_idx[-keep:]
    elif file_state == 3:
        if n <= keep:
            return windows_idx
        s = (n - keep) // 2
        return windows_idx[s:s+keep]
    elif file_state == 5:
        # 앞에서 target_n만큼 자르되, 부족하면 있는 만큼만 반환
        return windows_idx[:keep]
    return windows_idx

# arousal 파일만 통과시키는 필터
def is_arousal_file(path: Path) -> bool:
    name = path.stem.lower()
    # 포함 조건: 'arousal' 토큰이 있어야 함
    if "arousal" not in name:
        return False
    # 제외 조건: rest 관련 토큰들 포함 시 제외 (필요시 추가)
    exclude_tokens = ["rest", "resting", "space", "baseline", "calib", "practice"]
    return not any(tok in name for tok in exclude_tokens)

In [3]:
import numpy as np
import pandas as pd
import neurokit2 as nk

def clean_signals_with_neurokit(ppg_raw: np.ndarray, gsr_raw: np.ndarray, fs: int):
    """
    PPG: neurokit2 ppg_clean → 'ppg' 라벨로 반환
    EDA: eda_clean + eda_phasic → scl/scr
    """
    # PPG clean
    ppg_raw = np.asarray(ppg_raw, dtype=float)
    ppg = nk.ppg_clean(ppg_raw, sampling_rate=fs, method="elgendi")

    # EDA clean
    gsr = np.asarray(gsr_raw, dtype=float)
    if np.any(~np.isfinite(gsr)):
        m = np.nanmean(gsr)
        gsr = np.nan_to_num(gsr, nan=(0.0 if not np.isfinite(m) else float(m)))
    eda_clean = nk.eda_clean(gsr, sampling_rate=fs, method="neurokit")

    # tonic / phasic
    decomp, _ = nk.eda_phasic(eda_clean, sampling_rate=fs, method="neurokit")
    if isinstance(decomp, pd.DataFrame):
        scl = decomp["EDA_Tonic"].to_numpy()
        scr = decomp["EDA_Phasic"].to_numpy()
    elif isinstance(decomp, dict):
        scl = np.asarray(decomp["EDA_Tonic"])
        scr = np.asarray(decomp["EDA_Phasic"])
    else:
        signals, _ = nk.eda_process(gsr, sampling_rate=fs, method="neurokit")
        scl = signals["EDA_Tonic"].to_numpy()
        scr = signals["EDA_Phasic"].to_numpy()

    return ppg, scl, scr


In [4]:
def process_one_file_windows(path: Path) -> tuple[np.ndarray, np.ndarray]:
    """
    단일 파일 → 윈도 분할
    - X: [N, 640, 4]  (ppg, scl, scr, post_arousal)
    - y: [N]          (label = content_id)
    """
    df0 = read_two_header_as_standard_df(str(path))
    df = df0[["ppg", "gsr"]].copy()

    # 신호 정리
    ppg, scl, scr = clean_signals_with_neurokit(df["ppg"].to_numpy(), df["gsr"].to_numpy(), fs=FS)
    sig = np.stack([ppg, scl, scr], axis=1)  # [T, 3]

    # 윈도 인덱스
    WIN = int(round(WIN_SEC * FS))   # 640
    HOP = int(round(HOP_SEC * FS))   # 320
    n = len(sig)
    idx_windows = [(s, s+WIN) for s in range(0, max(0, n - WIN + 1), HOP)]

    # 메타에서 라벨 추출
    meta = _parse_filename_meta(str(path))
    idx_windows = trim_windows_by_state(idx_windows, int(meta.get("file_state", 0)), target_n=25)

    content_id   = int(meta.get("content_id", 0))      # y에 쓸 값
    post_arousal = int(meta.get("post_arousal", 0))    # X에 넣을 값

    X_list = []
    for s, e in idx_windows:
        seg = sig[s:e]
        if seg.shape[0] != WIN or not np.isfinite(seg).all():
            continue

        # post_arousal을 채널로 붙이기 → 모든 timestep에 같은 값
        post_arr = np.full((WIN, 1), post_arousal, dtype=np.float32)
        seg_ext = np.concatenate([seg.astype(np.float32), post_arr], axis=1)  # (640,4)
        X_list.append(seg_ext)

    if not X_list:
        return np.empty((0, WIN, 4), dtype=np.float32), np.empty((0,), dtype=int)

    X = np.stack(X_list, axis=0)  # [N, WIN, 4]
    y = np.full((X.shape[0],), content_id, dtype=int)

    return X, y


In [5]:
# 1) 단일 파일 처리 (예시)
ex_file = Path("./raw_data/p25/p25_11_5_space_0_24_9.csv")
# 윈도 분할 + 라벨 생성
X, y = process_one_file_windows(ex_file)

print("파일:", ex_file.name)
print("윈도 shape:", X.shape)   # (N, 640, 3) 기대
print("라벨 shape:", y.shape)   # (N,)
print("라벨 값:", np.unique(y))

# 예시 윈도 하나를 csv 파일로 저장
if X.size:
    k = 2
    example = pd.DataFrame(X[k], columns=["ppg","scl","scr","post_arousal"])
    example.to_csv("example_window.csv", index=False)
    print("example_window.csv 파일로 저장 완료")


파일: p25_11_5_space_0_24_9.csv
윈도 shape: (6, 640, 4)
라벨 shape: (6,)
라벨 값: [24]
example_window.csv 파일로 저장 완료


In [6]:
def save_subject_npz(subject_id: str, root_dir: Path = ROOT_DIR):
    """
    pXX 폴더의 arousal csv만 읽어 X,y 생성 후
    OUT_DIR/features_pXX_arousal.npz 저장
    """
    subj_dir = root_dir / subject_id
    files = sorted(p for p in subj_dir.glob("*.csv") if is_arousal_file(p))
    if not files:
        print(f"[WARN] no arousal files for {subject_id}")
        X = np.empty((0, WIN, 4), dtype=np.float32)
        y = np.empty((0,), dtype=int)
    else:
        X_list, y_list = [], []
        for f in files:
            Xf, yf = process_one_file_windows(f)
            if Xf.size:
                X_list.append(Xf); y_list.append(yf)
        if X_list:
            X = np.concatenate(X_list, axis=0)
            y = np.concatenate(y_list, axis=0)

            # ✅ 여기서 z-score 적용 (ppg, scl, scr만)
            for ch in range(3):  # 0=ppg, 1=scl, 2=scr
                vals = X[..., ch]
                mu = vals.mean()
                sigma = vals.std()
                X[..., ch] = (vals - mu) / sigma
        else:
            X = np.empty((0, WIN, 4), dtype=np.float32)
            y = np.empty((0,), dtype=int)

    out_path = OUT_DIR / f"features_{subject_id}_arousal.npz"
    np.savez_compressed(
        out_path,
        X=X,
        y=y,
        info=np.array({
            "fs": FS,
            "win_sec": WIN_SEC,
            "overlap": OVERLAP,
            "channels": ["ppg", "scl", "scr", "post_arousal"],
            "label": "content_id",
            "subject": subject_id
        }, dtype=object)
    )
    print(f"[OK] {subject_id}: X{X.shape}, y{y.shape} → {out_path}")
    return out_path, X, y


In [7]:
# _, X, y = save_subject_npz("p01")

In [8]:
# X[0].shape

In [9]:
def build_dataset_all_subjects(num_subjects: int,
                               start_index: int = 1,
                               zpad: int = 2,
                               root_dir: Path = ROOT_DIR):
    """
    p{start_index:0zpad} .. p{start_index+num_subjects-1:0zpad}
    각 피험자 NPZ 저장 + 전체 합본 NPZ 저장
    """
    subjects = [f"p0{i}" if i < 10 else f"p{i}"
            for i in range(start_index, start_index + num_subjects)]
    X_all_list, y_all_list, subj_ids = [], [], []

    for sid in subjects:
        _, X, y = save_subject_npz(sid, root_dir=root_dir)
        if X.size:
            X_all_list.append(X)
            y_all_list.append(y)
            subj_ids.append(np.full((X.shape[0],), sid, dtype=object))

    if not X_all_list:
        raise ValueError("No valid windows across all subjects.")

    X_all = np.concatenate(X_all_list, axis=0)
    y_all = np.concatenate(y_all_list, axis=0)
    subject_ids = np.concatenate(subj_ids, axis=0)

    # ⬇️ 여기 고침: 마지막 subject를 직접 뽑아서 파일명에 씀
    all_path = OUT_DIR / f"features_{subjects[0]}_to_{subjects[-1]}_arousal.npz"

    np.savez_compressed(
        all_path,
        X=X_all,
        y=y_all,
        subject_ids=subject_ids,
        info=np.array({
            "fs": FS,
            "win_sec": WIN_SEC,
            "overlap": OVERLAP,
            "channels": ["ppg", "scl", "scr", "post_arousal"],
            "label": "content_id",
            "subjects": subjects
        }, dtype=object)
    )
    print(f"[ALL] X{X_all.shape}, y{y_all.shape} → {all_path}")
    return all_path, X_all, y_all, subject_ids


In [10]:
all_path, X_all, y_all, subject_ids = build_dataset_all_subjects(num_subjects=54, start_index=1, zpad=2)

[OK] p01: X(452, 640, 4), y(452,) → dataset\dl_dataset\features_p01_arousal.npz
[OK] p02: X(445, 640, 4), y(445,) → dataset\dl_dataset\features_p02_arousal.npz
[OK] p03: X(446, 640, 4), y(446,) → dataset\dl_dataset\features_p03_arousal.npz
[OK] p04: X(443, 640, 4), y(443,) → dataset\dl_dataset\features_p04_arousal.npz
[OK] p05: X(447, 640, 4), y(447,) → dataset\dl_dataset\features_p05_arousal.npz
[OK] p06: X(446, 640, 4), y(446,) → dataset\dl_dataset\features_p06_arousal.npz
[OK] p07: X(453, 640, 4), y(453,) → dataset\dl_dataset\features_p07_arousal.npz
[OK] p08: X(449, 640, 4), y(449,) → dataset\dl_dataset\features_p08_arousal.npz
[OK] p09: X(459, 640, 4), y(459,) → dataset\dl_dataset\features_p09_arousal.npz
[OK] p10: X(448, 640, 4), y(448,) → dataset\dl_dataset\features_p10_arousal.npz
[OK] p11: X(448, 640, 4), y(448,) → dataset\dl_dataset\features_p11_arousal.npz
[OK] p12: X(450, 640, 4), y(450,) → dataset\dl_dataset\features_p12_arousal.npz
[OK] p13: X(445, 640, 4), y(445,) → data

c:\Users\jaebb\anaconda3\envs\steam\lib\site-packages\neurokit2\eda\eda_peaks.py:127: RuntimeWarning: All-NaN slice encountered
  info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"])


[OK] p25: X(453, 640, 4), y(453,) → dataset\dl_dataset\features_p25_arousal.npz
[OK] p26: X(449, 640, 4), y(449,) → dataset\dl_dataset\features_p26_arousal.npz
[OK] p27: X(452, 640, 4), y(452,) → dataset\dl_dataset\features_p27_arousal.npz
[OK] p28: X(452, 640, 4), y(452,) → dataset\dl_dataset\features_p28_arousal.npz
[OK] p29: X(450, 640, 4), y(450,) → dataset\dl_dataset\features_p29_arousal.npz
[OK] p30: X(452, 640, 4), y(452,) → dataset\dl_dataset\features_p30_arousal.npz
[OK] p31: X(455, 640, 4), y(455,) → dataset\dl_dataset\features_p31_arousal.npz
[OK] p32: X(456, 640, 4), y(456,) → dataset\dl_dataset\features_p32_arousal.npz
[OK] p33: X(452, 640, 4), y(452,) → dataset\dl_dataset\features_p33_arousal.npz
[OK] p34: X(451, 640, 4), y(451,) → dataset\dl_dataset\features_p34_arousal.npz
[OK] p35: X(451, 640, 4), y(451,) → dataset\dl_dataset\features_p35_arousal.npz
[OK] p36: X(455, 640, 4), y(455,) → dataset\dl_dataset\features_p36_arousal.npz
[OK] p37: X(453, 640, 4), y(453,) → data

In [11]:
import numpy as np
import pandas as pd
from pathlib import Path

def load_subject_npz(path: Path):
    """features_pXX_arousal.npz -> (X:[N,640,4], y:[N], info:dict)"""
    if not Path(path).exists():
        raise FileNotFoundError(path)
    data = np.load(path, allow_pickle=True)
    X = data["X"]
    y = data["y"]
    info = data["info"].item() if "info" in data else {}
    return X, y, info

def load_all_npz(path: Path):
    """features_p01_to_pNN_arousal.npz -> (X:[N,640,4], y:[N], subject_ids:[N], info:dict)"""
    if not Path(path).exists():
        raise FileNotFoundError(path)
    data = np.load(path, allow_pickle=True)
    X = data["X"]
    y = data["y"]
    subject_ids = data["subject_ids"] if "subject_ids" in data else None
    info = data["info"].item() if "info" in data else {}
    return X, y, subject_ids, info

def summarize_dataset(X, y, subject_ids=None, info=None, max_rows=10):
    ch = info.get("channels", ["ppg","scl","scr","post_arousal"]) if info else ["ppg","scl","scr","post_arousal"]
    print("X shape:", X.shape)  # (N, 640, 4)
    print("y shape:", y.shape)
    if subject_ids is not None:
        print("subject_ids shape:", subject_ids.shape)
        # 앞부분 몇 개만 출력
        print("subject_ids head:", subject_ids[:min(10, len(subject_ids))])
    uniq, cnt = np.unique(y, return_counts=True)
    print("y unique:", uniq)
    print("y distribution:", dict(zip(uniq, cnt)))
    print("channels:", ch)
    # 예시 윈도 하나
    if X.size:
        k = 0
        df = pd.DataFrame(X[k], columns=ch)
        display(df.head(max_rows))
        print("post_arousal(unique in window k=0):", pd.unique(df["post_arousal"]))

In [12]:
all_path = Path("./dataset/dl_dataset/features_p01_to_p54_arousal.npz")  # 경로만 맞춰줘
X_all, y_all, subject_ids, info_all = load_all_npz(all_path)

print("=== ALL SUMMARY ===")
summarize_dataset(X_all, y_all, subject_ids=subject_ids, info=info_all)

# (옵션) 특정 subject / 특정 content_id로 예시 윈도 골라 보기
target_subject = "p07"          # 바꿔서 사용
target_cid     = None           # 예: 4 로 지정하면 그 라벨만 필터

idx = np.arange(len(y_all))
if subject_ids is not None:
    idx = idx[subject_ids == target_subject]
if target_cid is not None:
    idx = idx[y_all[idx] == target_cid]

if idx.size:
    k = idx[0]
    cols = info_all.get("channels", ["ppg","scl","scr","post_arousal"])
    ex = pd.DataFrame(X_all[k], columns=cols)
    print(f"Example window @ k={k}, subject={subject_ids[k] if subject_ids is not None else 'NA'}, label(y)={y_all[k]}")
    display(ex.head(10))
    print("post_arousal(unique):", np.unique(X_all[k, :, cols.index("post_arousal")]))
else:
    print("해당 조건에 맞는 윈도가 없습니다.")

=== ALL SUMMARY ===
X shape: (24345, 640, 4)
y shape: (24345,)
subject_ids shape: (24345,)
subject_ids head: ['p01' 'p01' 'p01' 'p01' 'p01' 'p01' 'p01' 'p01' 'p01' 'p01']
y unique: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53]
y distribution: {np.int64(0): np.int64(455), np.int64(1): np.int64(451), np.int64(2): np.int64(448), np.int64(3): np.int64(452), np.int64(4): np.int64(448), np.int64(5): np.int64(451), np.int64(6): np.int64(454), np.int64(7): np.int64(451), np.int64(8): np.int64(455), np.int64(9): np.int64(450), np.int64(10): np.int64(449), np.int64(11): np.int64(453), np.int64(12): np.int64(453), np.int64(13): np.int64(453), np.int64(14): np.int64(452), np.int64(15): np.int64(450), np.int64(16): np.int64(450), np.int64(17): np.int64(452), np.int64(18): np.int64(448), np.int64(19): np.int64(458), np.int64(20): np.int64(451), np.int64(21): np.int64(449), np.int64

,ppg,scl,scr,post_arousal
0,-0.178676,-0.246284,0.018418,0.0
1,-0.172000,-0.246302,0.017997,0.0
2,-0.169513,-0.246318,0.017585,0.0
3,-0.174931,-0.246335,0.017187,0.0
4,-0.190976,-0.246353,0.016802,0.0
5,-0.219087,-0.246369,0.016432,0.0
6,-0.259308,-0.246387,0.016081,0.0
7,-0.310368,-0.246404,0.015747,0.0
8,-0.369910,-0.246421,0.015434,0.0
9,-0.434817,-0.246438,0.015141,0.0


post_arousal(unique in window k=0): [0.]
Example window @ k=2679, subject=p07, label(y)=0


,ppg,scl,scr,post_arousal
0,0.206571,-0.704454,-0.429307,0.0
1,0.077107,-0.704388,-0.430374,0.0
2,-0.044708,-0.704322,-0.431390,0.0
3,-0.152121,-0.704255,-0.432347,0.0
4,-0.239648,-0.704189,-0.433238,0.0
5,-0.303368,-0.704122,-0.434058,0.0
6,-0.340979,-0.704055,-0.434799,0.0
7,-0.351659,-0.703989,-0.435457,0.0
8,-0.335807,-0.703922,-0.436029,0.0
9,-0.294750,-0.703855,-0.436512,0.0


post_arousal(unique): [0.]
